# 🧗 Climbing Hold Detection - YOLOv8 Fine-tuning

Roboflow 데이터셋으로 YOLOv8s 모델을 파인튜닝합니다.

**사전 준비:**
1. 런타임 → 런타임 유형 변경 → **GPU (T4)** 선택
2. Roboflow에서 데이터셋 ZIP 다운로드 (YOLOv8 포맷)

**Colab 무료 제한:**
- GPU 사용 시간 제한 있음 (보통 4~12시간)
- 초과해도 과금 없음, 그냥 GPU 할당 안 됨
- 학습 100 epoch 기준 약 1~2시간 소요

## 1. 설치

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
# GPU 확인
!nvidia-smi

## 2. 데이터셋 로드

Google Drive에 데이터셋을 넣어두고 마운트해서 사용합니다.
(ZIP 또는 압축 해제된 폴더 모두 OK)

In [ ]:
import os
import subprocess
from google.colab import drive

# Google Drive 마운트
drive.mount("/content/drive")

# Drive에서 data.yaml 파일 검색 (압축 해제된 폴더)
result = subprocess.run(
    ["find", "/content/drive/MyDrive", "-maxdepth", "5", "-name", "data.yaml", "-type", "f"],
    capture_output=True, text=True, timeout=60
)
found_yamls = [line for line in result.stdout.strip().split("\n") if line]
climb_yamls = [y for y in found_yamls if any(kw in y.lower() for kw in ["climb", "hold", "neural"])]

# ZIP 파일도 검색
result2 = subprocess.run(
    ["find", "/content/drive/MyDrive", "-maxdepth", "3", "-name", "*.zip", "-type", "f"],
    capture_output=True, text=True, timeout=30
)
climb_zips = [z for z in result2.stdout.strip().split("\n")
              if z and any(kw in z.lower() for kw in ["climb", "hold", "neural"])]

if climb_yamls:
    print("Found datasets (folders):")
    for y in climb_yamls:
        print(f"  {y}")
elif climb_zips:
    print("Found ZIP files:")
    for z in climb_zips:
        print(f"  {z}")
else:
    print("No climbing datasets found. Check your Drive folder structure.")

In [ ]:
import shutil

# 폴더인 경우: Drive에서 로컬로 복사 (Drive 직접 읽으면 느림)
if climb_yamls:
    for i, yaml_path in enumerate(climb_yamls):
        ds_dir = os.path.dirname(yaml_path)
        local_dir = f"datasets/dataset_{i}"
        print(f"\nCopying {ds_dir} → {local_dir}/ ...")
        shutil.copytree(ds_dir, local_dir, dirs_exist_ok=True)
        print("Done")

# ZIP인 경우: 압축 해제
elif climb_zips:
    for i, zip_path in enumerate(climb_zips):
        local_dir = f"datasets/dataset_{i}"
        print(f"\nExtracting {os.path.basename(zip_path)} → {local_dir}/")
        os.makedirs(local_dir, exist_ok=True)
        !unzip -q "{zip_path}" -d "{local_dir}"
        print("Done")

# 확인
!find datasets/ -name "data.yaml" -type f

## 3. 데이터셋 준비

라벨을 단일 `hold` 클래스로 통합하고 data.yaml을 설정합니다.

In [ ]:
import yaml
import os
import shutil

# 모든 데이터셋을 하나로 병합
merged_dir = "datasets/merged"
for split in ["train", "valid"]:
    os.makedirs(f"{merged_dir}/{split}/images", exist_ok=True)
    os.makedirs(f"{merged_dir}/{split}/labels", exist_ok=True)

# 각 데이터셋에서 파일 복사
yaml_paths = []
for root, dirs, fnames in os.walk("datasets"):
    if "data.yaml" in fnames and "merged" not in root:
        yaml_paths.append(os.path.join(root, "data.yaml"))

print(f"Found {len(yaml_paths)} dataset(s)")

for yaml_path in yaml_paths:
    ds_dir = os.path.dirname(yaml_path)
    with open(yaml_path) as f:
        config = yaml.safe_load(f)
    print(f"  {yaml_path}: classes={config.get('names', {})}")

    for split in ["train", "valid", "val", "test"]:
        target_split = "valid" if split in ("valid", "val", "test") else "train"
        for subdir in ["images", "labels"]:
            src_dir = os.path.join(ds_dir, split, subdir)
            if not os.path.isdir(src_dir):
                continue
            dst_dir = f"{merged_dir}/{target_split}/{subdir}"
            for fname in os.listdir(src_dir):
                src = os.path.join(src_dir, fname)
                dst = os.path.join(dst_dir, fname)
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)

# 모든 라벨을 class 0 (hold)로 통합
remapped = 0
for split in ["train", "valid"]:
    labels_dir = f"{merged_dir}/{split}/labels"
    for fname in os.listdir(labels_dir):
        if not fname.endswith(".txt"):
            continue
        fpath = os.path.join(labels_dir, fname)
        lines = []
        with open(fpath) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    parts[0] = "0"
                    lines.append(" ".join(parts))
        with open(fpath, "w") as f:
            f.write("\n".join(lines) + "\n" if lines else "")
        remapped += 1

# data.yaml 생성
yaml_path = f"{merged_dir}/data.yaml"
merged_config = {
    "path": os.path.abspath(merged_dir),
    "train": "train/images",
    "val": "valid/images",
    "names": {0: "hold"},
}
with open(yaml_path, "w") as f:
    yaml.dump(merged_config, f, default_flow_style=False)

train_count = len([f for f in os.listdir(f"{merged_dir}/train/images") if not f.startswith(".")])
val_count = len([f for f in os.listdir(f"{merged_dir}/valid/images") if not f.startswith(".")])
print(f"\nMerged: {train_count} train, {val_count} val images")
print(f"Remapped {remapped} label files → single 'hold' class")

## 4. YOLOv8s 파인튜닝

- **모델**: YOLOv8s (small, 11.2M params) — nano 대비 mAP ~20% 향상
- **epochs**: 100 (early stopping patience=20)
- **예상 시간**: T4 GPU에서 1~2시간

In [ ]:
from ultralytics import YOLO

# YOLOv8s 프리트레인 모델 로드
model = YOLO("yolov8s.pt")

# 학습 시작
results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,          # OOM 나면 8로 줄이기
    patience=20,       # 20 epoch 개선 없으면 조기 종료
    project="runs/hold_detect",
    name="yolov8s",
    save=True,
    plots=True,
    verbose=True,
)

## 5. 결과 확인

In [ ]:
# 학습 결과 확인
best_model = YOLO("runs/hold_detect/yolov8s/weights/best.pt")
metrics = best_model.val(data=f"{merged_dir}/data.yaml", verbose=True)

print("\n=== 📊 Results ===")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

In [ ]:
# 학습 곡선 시각화
from IPython.display import Image, display

display(Image(filename="runs/hold_detect/yolov8s/results.png", width=800))

In [ ]:
# 샘플 예측 결과 시각화
display(Image(filename="runs/hold_detect/yolov8s/val_batch0_pred.jpg", width=800))

## 6. 모델 다운로드

학습된 모델을 다운로드하여 프로젝트에 적용합니다.

1. 아래 셀에서 모델 다운로드
2. 프로젝트의 `backend/models/` 에 복사
3. `.env` 파일에 `HOLD_MODEL_PATH=models/yolov8s-climbing-holds.pt` 추가
4. 서버 재시작

In [ ]:
# Google Drive에 저장 (Colab 세션 종료 대비)
from google.colab import drive
drive.mount("/content/drive")

save_path = "/content/drive/MyDrive/climbing-models/yolov8s-climbing-holds.pt"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
shutil.copy2("runs/hold_detect/yolov8s/weights/best.pt", save_path)
print(f"✅ Model saved to Google Drive: {save_path}")

In [ ]:
# 또는 직접 다운로드
from google.colab import files
files.download("runs/hold_detect/yolov8s/weights/best.pt")